In [1]:
import sys
#lets you access python's runtime environment
from pathlib import Path
#sys.path is a built in variable in the sys module and contains a list of directories that is seached through when you do an import
#so we are appending the src directory to that
sys.path.append(str(Path().resolve().parent / "src"))
import config
from sklearn.model_selection import GridSearchCV, BaseCrossValidator
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
panel_data_path = config.PROJECT_ROOT/ "data" / "panel_data.csv"
panel_data = pd.read_csv(panel_data_path)

features = ['TimeSinceLastPBYearEnd',
 'BestGoodliftOfYear',
 'ImprovementGradientWithinYear',
 'Age',
 'AvgMeetsPerYear',
 'Sex']

#Year needed for timeaware split
panel = panel_data.loc[:,features + ['Churns'] +['Year']]


In [2]:

train = panel[panel['Year']<=2023].copy()
test = panel[panel['Year']>2023].copy()

train_X = train.drop(columns = 'Churns')
train_y = train['Churns']

test_X = test.drop(columns = 'Churns')
test_y = test['Churns']

Note that early stopping was tried but model performed better without so will tune max_iter instead in GridSearchCV

In [3]:
class YearBasedTimeSeriesSplit(BaseCrossValidator):
    #see https://github.com/scikit-learn/scikit-learn/blob/main/sklearn/model_selection/_split.py
    # inheriting from BaseCrossValidator to ensure compatibility with GridSearchCV
    
    def __init__(self, year_column='Year', exclude_val_years = None):
        self.year_column = year_column
        self.exclude_val_years = exclude_val_years or ()

    def split(self, X, y=None, groups=None):
        #needed to overwrite split method in parent class to make sure splits are time aware
        #otherwise the training rows are just all rows that are not validation rows which would create leakage in hyperparam tuning
        
        years = sorted(X[self.year_column].unique())
        
        for i in range(1, len(years)):
            val_year = years[i]
            if val_year in self.exclude_val_years:
                continue
            train_years = years[:i]
            train_idx = np.where(X[self.year_column].isin(train_years))[0]
            val_idx = np.where(X[self.year_column] == val_year)[0]

            #parent class uses yield in implementation of split() so will use here
            yield train_idx, val_idx
        
    def _iter_test_indices(self, X=None, y=None, groups=None):
        #BaseCrossValidator expects _iter_test_indices or _iter_test_masks
        #but since split had to be overwritten, logic for _iter_test_indices is same as for split().
        #(usually split calls _iter_test_masks which calls _iter_test_indices but here it makes more sense to call split for _iter_test_indices)
        for _, val_idx in self.split(X):
            yield val_idx
            
    def get_n_splits(self, X=None, y=None, groups=None):
        #returns the number of folds the splitter will generate. 
        years = X[self.year_column].unique()
        #checks which years in excluded_years are actually present in the dataset
        excl_length = len([y for y in years if y in self.exclude_val_years])
        return len(years) - 1 - excl_length

In [4]:
year_cv = YearBasedTimeSeriesSplit(year_column='Year')

param_grid = {
    'learning_rate': [0.01, 0.05, 0.1],
    'max_iter': [200, 500, 1000],
    'max_depth': [3, 6, 9],
    'min_samples_leaf': [10, 50],
    'l2_regularization': [0.0, 0.1]
}
grid_search = GridSearchCV(
    HistGradientBoostingClassifier(random_state=42, categorical_features = ['Sex']),
    param_grid,
    scoring='accuracy',
    cv=year_cv,
    n_jobs=-1,
    verbose=1
)


grid_search.fit(train_X, train_y)

Fitting 8 folds for each of 108 candidates, totalling 864 fits


C:\Users\bnpar\miniconda3\envs\powerlifting\Lib\site-packages\joblib\externals\loky\process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",HistGradientB...ndom_state=42)
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'l2_regularization': [0.0, 0.1], 'learning_rate': [0.01, 0.05, ...], 'max_depth': [3, 6, ...], 'max_iter': [200, 500, ...], ...}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'accuracy'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",YearBasedTime...column='Year')
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the com

In [5]:
results_df = pd.DataFrame(grid_search.cv_results_)
best_max_iter = grid_search.best_params_['max_iter']
best_learning_rate = grid_search.best_params_['learning_rate']
best_max_depth = grid_search.best_params_['max_depth']
best_min_samples_leaf = grid_search.best_params_['min_samples_leaf']
best_l2_regularization = grid_search.best_params_['l2_regularization']

best_params = {
    'learning_rate': best_learning_rate,
    'max_iter': best_max_iter,
    'max_depth': best_max_depth,
    'min_samples_leaf': best_min_samples_leaf,
    'l2_regularization': best_l2_regularization
}
best_params

{'learning_rate': 0.01,
 'max_iter': 500,
 'max_depth': 3,
 'min_samples_leaf': 10,
 'l2_regularization': 0.1}

In [6]:
results_df.loc[results_df['rank_test_score']==1, 'mean_test_score']

56    0.650584
Name: mean_test_score, dtype: float64

In [7]:
fine_param_grid = {
    'learning_rate': [0.005,0.01, 0.03],
    'max_iter': [350, 500, 750],
    'max_depth': [2,3,4],
    'min_samples_leaf': [5,10,30],
    'l2_regularization': [0.05, 0.1,1]
}




grid_search_fine = GridSearchCV(
    HistGradientBoostingClassifier(random_state=42, categorical_features = ['Sex']),
    fine_param_grid,
    scoring='accuracy',
    cv=year_cv,
    n_jobs=-1,
    verbose=1
)

grid_search_fine.fit(train_X, train_y)

Fitting 8 folds for each of 243 candidates, totalling 1944 fits


C:\Users\bnpar\miniconda3\envs\powerlifting\Lib\site-packages\joblib\externals\loky\process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",HistGradientB...ndom_state=42)
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'l2_regularization': [0.05, 0.1, ...], 'learning_rate': [0.005, 0.01, ...], 'max_depth': [2, 3, ...], 'max_iter': [350, 500, ...], ...}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'accuracy'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",YearBasedTime...column='Year')
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : 

In [8]:
results_df_fine = pd.DataFrame(grid_search_fine.cv_results_)
best_max_iter_fine = grid_search_fine.best_params_['max_iter']
best_learning_rate_fine = grid_search_fine.best_params_['learning_rate']
best_max_depth_fine = grid_search_fine.best_params_['max_depth']
best_min_samples_leaf_fine = grid_search_fine.best_params_['min_samples_leaf']
best_l2_regularization_fine = grid_search_fine.best_params_['l2_regularization']

best_params_fine = {
    'learning_rate': best_learning_rate_fine,
    'max_iter': best_max_iter_fine,
    'max_depth': best_max_depth_fine,
    'min_samples_leaf': best_min_samples_leaf_fine,
    'l2_regularization': best_l2_regularization_fine
}
best_params_fine

{'learning_rate': 0.01,
 'max_iter': 750,
 'max_depth': 2,
 'min_samples_leaf': 5,
 'l2_regularization': 0.05}

In [9]:
results_df_fine.loc[results_df_fine['rank_test_score']==1, 'mean_test_score']

33    0.651183
Name: mean_test_score, dtype: float64

In [10]:
final_params = {'learning_rate': 0.01,
 'max_iter': 750,
 'max_depth': 2,
 'min_samples_leaf': 5,
 'l2_regularization': 0.05}

Optimal parameters after first grid search has mean accuracy of 0.65112 across folds. After second grid search has mean accuracy of 0.651376. Accuracy not changing much so will terminate grid search. Model evaluation can be seen in test.ipynb